In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os

# Load Dataset

In [13]:
df = pd.read_csv("../data/dataset_pupuk.csv")
# df = pd.read_csv("../data/sintetis/dataset_pupuk.csv")
print(f"Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Dataset: 257 baris, 8 kolom


,nitrogen,fosfor,kalium,plant_age,fase,ec,soil_moisture,recommendation
0,128,232,163,20,0,38.29,23.7,10
1,128,232,163,20,0,38.31,23.5,10
2,127,232,163,20,0,38.33,23.3,10
3,126,232,162,20,0,38.35,23.2,10
4,125,232,162,20,0,38.36,23.0,10


# Definisi fitur dan label

In [14]:
FEATURES = ["nitrogen", "fosfor", "kalium", "plant_age", "fase",
            "ec", "soil_moisture"]

LABEL_NAMES = {
    0: "Tidak perlu",
    1: "Urea/ZA",
    2: "SP-36",
    3: "KCl",
    4: "Urea/ZA + SP-36",
    5: "Urea/ZA + KCl",
    6: "SP-36 + KCl",
    7: "Urea/ZA + SP-36 + KCl",
    8: "NPK 15-15-15",
    9: "Kurangi pemupukan N",
    10: "Flush air (EC/nutrisi tinggi)",
}

X = df[FEATURES].values
y = df["recommendation"].values

# Cek Distribusi Label (KRITIS)

In [15]:
print(f"Jumlah kelas unik: {len(set(y))}")
print(f"\nDistribusi label:")
dist = pd.Series(y).value_counts().sort_index()
for val, count in dist.items():
    print(f"  {val} = {LABEL_NAMES.get(val, '?'):35s} {count:5d} ({count/len(y)*100:.1f}%)")

if len(set(y)) < 2:
    print("\n⚠️  BAHAYA: Cuma 1 kelas! Model gak bisa dilatih.")
    print("    Data kamu perlu variasi kondisi NPK/fase lebih banyak.")

Jumlah kelas unik: 1

Distribusi label:
  10 = Flush air (EC/nutrisi tinggi)         257 (100.0%)

⚠️  BAHAYA: Cuma 1 kelas! Model gak bisa dilatih.
    Data kamu perlu variasi kondisi NPK/fase lebih banyak.


# Split Train/Test

In [16]:
# stratify butuh tiap kelas minimal 2 sampel; kalau error, hapus stratify
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    print("Stratify gagal (ada kelas <2 sampel), pakai split biasa.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print(f"Train: {len(X_train)} baris")
print(f"Test:  {len(X_test)} baris")

Train: 205 baris
Test:  52 baris


# Training Random Forest

In [17]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print("Training selesai.")

Training selesai.


# Cross Validation

In [18]:
n_splits = min(5, pd.Series(y_train).value_counts().min())
if n_splits < 2:
    print("Data per kelas terlalu sedikit untuk cross-validation.")
else:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_weighted")
    print(f"CV F1 (weighted): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV F1 (weighted): 1.0000 (+/- 0.0000)


# Evaluasi

In [19]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
print()
labels_present = sorted(set(y_test) | set(y_pred))
target_names = [LABEL_NAMES.get(l, str(l)) for l in labels_present]
print(classification_report(y_test, y_pred, labels=labels_present,
                            target_names=target_names, zero_division=0))

Accuracy: 1.0000
F1 (weighted): 1.0000

                               precision    recall  f1-score   support

Flush air (EC/nutrisi tinggi)       1.00      1.00      1.00        52

                     accuracy                           1.00        52
                    macro avg       1.00      1.00      1.00        52
                 weighted avg       1.00      1.00      1.00        52



# Feature Importance

In [20]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print("Feature Importance:")
for idx in sorted_idx:
    bar = "█" * int(importances[idx] * 50)
    print(f"  {FEATURES[idx]:15s} {importances[idx]:.4f}  {bar}")

Feature Importance:
  soil_moisture   0.0000  
  ec              0.0000  
  fase            0.0000  
  plant_age       0.0000  
  kalium          0.0000  
  fosfor          0.0000  
  nitrogen        0.0000  


# Tes Prediksi

In [21]:
sample = {
    "nitrogen": 40,
    "fosfor": 40,
    "kalium": 100,
    "plant_age": 61,
    "fase": 1,
    "ec": 2.0,
    "soil_moisture": 65,
}
X_sample = np.array([[sample[f] for f in FEATURES]])
pred = int(model.predict(X_sample)[0])
conf = float(model.predict_proba(X_sample)[0].max())

print(f"Prediksi: {pred} = {LABEL_NAMES.get(pred)}")
print(f"Confidence: {conf:.2%}")


Prediksi: 10 = Flush air (EC/nutrisi tinggi)
Confidence: 100.00%


# Simpan Model

In [22]:
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/rf_pupuk.joblib")
print("Model tersimpan: models/rf_pupuk.joblib")

Model tersimpan: models/rf_pupuk.joblib
